In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from matplotlib.colors import ListedColormap

In [ ]:
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
sc.settings.verbosity = 3 
sc.logging.print_header()
sc.settings.set_figure_params(dpi=300, transparent = True, format = 'pdf', vector_friendly = True)

In [ ]:
figure = "Figure_2-3"

In [ ]:
sc.settings.figdir = './Figure_plots/'+figure + '/2609/'

In [ ]:
umap_cmap = sns.blend_palette(['xkcd:light grey', 'xkcd:blueberry'], as_cmap = True)

In [ ]:
# ---- Build color palette + colormap ----
def truncate_colormap(cmap, minval=0, maxval=1.0, n=256):
    """Remove darkest colors from a colormap."""
    return ListedColormap(cmap(np.linspace(minval, maxval, n)))

# Declaring the input files

In [ ]:
adata = sc.read_h5ad('./h5ad/analysis_250528_f/Smed_L78-L47_20250523_Annotated.h5ad')

In [ ]:
adata

In [ ]:
samp = 'Sample'

In [ ]:
clusteringlayer = 'annotated_names'

# Plots

## barplots

In [ ]:
adata.obs['neoblast_cat'] = 'low'
adata.obs.loc[(adata.obs['neoblast_score'] > 0.185), 'neoblast_cat'] = 'high'

In [ ]:
categories = adata.obs["annotated_names"].cat.categories
colors = adata.uns["annotated_names_colors"]

In [ ]:
df = adata.obs.copy()

# Calculate percentage of 'high' in each annotated_name
pct_high = (
    adata.obs.groupby("annotated_names")["neoblast_cat"]
    .apply(lambda x: (x == "high").mean() * 100)  # fraction to %
    .reset_index(name="percent_high")
)



In [ ]:
adata.obs[[clusteringlayer, 'Sample_2C/4C']]

In [ ]:
cluster_col = clusteringlayer
sample_col = 'Sample_2C/4C'

In [ ]:
df = adata.obs[[cluster_col, sample_col]]

In [ ]:
pivot_counts = pd.crosstab(df[cluster_col], df[sample_col])

In [ ]:
pivot_counts

In [ ]:
pivot_percent = pivot_counts.div(pivot_counts.sum(axis=1), axis=0) * 100

In [ ]:
pivot_percent

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(20, 10), sharex=True)

sc.pl.violin(adata, keys = 'neoblast_score', groupby = clusteringlayer, stripplot = False, show = False, ax = axes[0])

sns.barplot(data=pct_high, x="annotated_names", y="percent_high", edgecolor="black", linewidth=1, palette=colors, ax = axes[1])

sns.barplot(data=pivot_percent, x="annotated_names", y="4C", edgecolor="black", linewidth=1, palette=colors, ax = axes[2])

axes[2].tick_params(axis='x', labelrotation=90)
plt.tight_layout()

#plt.savefig('./Figure_plots/'+ figure + '/' + figure + "_violin_barplot.pdf")

plt.show()



# metacell

In [ ]:
meta = pd.read_csv("metacell_analysis/smed_cdh1_3k.mc2.csv")
meta = meta.set_index('Unnamed: 0')
meta

In [ ]:
meta.columns

In [ ]:
adata.obs['metacell'] = meta['mc_name']
adata.obs['metacell'] = adata.obs['metacell'].astype('category')

In [ ]:
adata.obs['metacell']

In [ ]:
adata.obs['metacell'].value_counts().mean()

In [ ]:
majority = adata.obs.groupby('metacell')['annotated_names'].agg(lambda x: x.value_counts().idxmax())
adata.obs['metacell_annotated'] = adata.obs['metacell'].map(majority)

In [ ]:
adata.uns['metacell_annotated_colors'] = adata.uns['annotated_names_colors']

In [ ]:
sc.pl.umap(adata, color = 'metacell_annotated')

In [ ]:
df_meta = adata.obs.groupby('metacell').agg(
    metacell_annotated=('metacell_annotated', 'first'),
    metacell_broad=('broad_names' , lambda x: x.mode()[0]),
    neoblast_score=('neoblast_score', 'mean')
)
df_meta

In [ ]:
counts_df = df_meta.groupby('metacell_annotated')['neoblast_score'].apply(
    lambda x: pd.Series({
        'high': (x > 0.258).sum(),
        'medium': ((x <= 0.258) & (x > 0.171)).sum(),
        'low': (x <= 0.171).sum()
    })
).unstack()

totals = counts_df.sum(axis=1)
pct_df = counts_df.div(totals, axis=0) * 100
pct_df = pct_df[['high', 'medium', 'low']]

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(24, 13), sharex=True)
colors = list(adata.uns["annotated_names_colors"])

# violin plot of neoblast score
sc.pl.violin(adata, keys='neoblast_score', groupby=clusteringlayer, stripplot=False, show=False, ax=axes[0])
axes[0].set_ylabel('Neoblast score (Nsc)')
axes[0].set_axisbelow(True)
axes[0].yaxis.grid(True)

# barplot of high percentage
sns.barplot(data=pct_high, x="annotated_names", y="percent_high", edgecolor="black", linewidth=1, palette=colors, ax=axes[1])
axes[1].set_ylabel('% high Nsc cells')
axes[1].set_axisbelow(True)
axes[1].yaxis.grid(True)

# barplot of neoblast scores in metacells
pct_df.plot(kind='bar', stacked=True, ax=axes[2], zorder=3, width=0.8, 
            color=['black','dimgrey', 'lightgrey'], edgecolor=colors + colors + colors, linewidth=0.75, legend=False)
axes[2].xaxis.grid(False)
axes[2].yaxis.grid(True, zorder=0)
axes[2].set_ylim(0, 110)
axes[2].set_ylabel('% metacells with\nhigh/medium/low Nsc')
handles, labels = axes[2].get_legend_handles_labels()
axes[2].legend(handles[::-1], labels[::-1], loc='upper right', fontsize=13, bbox_to_anchor=(1, 1))

for i, total in enumerate(totals):
    axes[2].annotate(str(total), xy=(i, 1.04), xycoords=('data', 'axes fraction'),
                      ha='center', fontsize=14, rotation=90, annotation_clip=False)

# barplot of 4C cells
sns.barplot(data=pivot_percent, x="annotated_names", y="4C", edgecolor="black", linewidth=1, palette=colors, ax=axes[3])
axes[3].tick_params(axis='x', labelrotation=90)
axes[3].set_ylabel('% 4C cells')
axes[3].set_axisbelow(True)
axes[3].yaxis.grid(True)


for ax in [axes[0], axes[1], axes[3]]:
    ax.set_axisbelow(True)
    ax.yaxis.grid(True)
    
plt.rcParams['font.family'] = 'Arial'
for ax in axes:
    ax.yaxis.label.set_fontsize(17)
    ax.yaxis.label.set_fontweight('bold')
plt.xticks(fontsize=15)
plt.xlabel('')

plt.subplots_adjust(hspace=0.12)
pos1 = axes[1].get_position()
pos2 = axes[2].get_position()
gap = 0.025
axes[2].set_position([pos2.x0, pos2.y0 - gap, pos2.width, pos2.height])
axes[3].set_position([axes[3].get_position().x0, axes[3].get_position().y0 - gap, axes[3].get_position().width, axes[3].get_position().height])

plt.savefig('./Figure_plots/'+ figure + '/' + figure + "_violin_barplot-2.pdf")
plt.show()

# subclustering umap plots and heatmaps

In [ ]:
resolutions = [1, 2, 3, 4, 5]

In [ ]:
pal = 'coolwarm'

In [ ]:
names = adata.obs['annotated_names'].cat.categories
colours_n = list(adata.uns[clusteringlayer + '_colors'])

# 4C subclustering analysis

In [ ]:
name_of_subclustering = '_4C_'
adata = sc.read_h5ad('./h5ad/analysis_250528_f/test subclustering 2609/Smed_L47_Subclustering_4C_Results_2608.h5ad')

In [ ]:
sc.pl.umap(adata, color = 'annotated_names',  legend_loc = None, frameon = False, title = 'Subclustering' + name_of_subclustering, 
           save = name_of_subclustering
          )

In [ ]:
results = pd.DataFrame(
    0,
    index = names,
    columns = resolutions,
    dtype = int
)

In [ ]:
for i in resolutions:
    for l in adata.obs['leiden'+ name_of_subclustering + str(i)].cat.categories:
        counts = adata.obs.loc[(adata.obs['leiden'+ name_of_subclustering + str(i)] == l), 'annotated_names'].value_counts().sort_values(ascending = False)
        if counts[0] > (counts[1:].sum() / 10):
            results.loc[counts.index[0], i] += 1

In [ ]:
cmap = LinearSegmentedColormap.from_list('custom', 
    [(0, (0,0,0)), (0.01, (0.4, 0.1, 0.5)), (0.15, (0.25, 0.4, 0.85)), (0.4, (0.45, 0.65, 0.95)), (1, (0.88, 0.91, 0.95))])

In [ ]:
fig, ax = plt.subplots(figsize=(13,30))
sns.heatmap(results.iloc[:-5, :],cmap = cmap, linewidths=1, linecolor='grey', ax=ax)
plt.grid(False)


for ticklabel, color in zip(ax.get_yticklabels(), colours_n):
    ticklabel.set_color(color)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=25)

plt.yticks(fontsize = 25)
plt.xticks(fontsize = 30)
    
plt.tight_layout()
plt.savefig('./Figure_plots/'+ figure + '/2609/' + figure + name_of_subclustering + "_heatmap.pdf")    
plt.show()

In [ ]:
with plt.rc_context({'figure.figsize': (2, 5)}):
    ax = sc.pl.violin(adata, keys='n_counts', size=0.5, jitter=0.4, show=False)
    ax.grid(zorder=0)
    ax.set_axisbelow(True)
    plt.axhline(y=adata.obs['n_counts'].mean(), color='r', linestyle='-')
    plt.savefig('./Figure_plots/'+ figure + '/2609/' + figure + name_of_subclustering + "_violin.pdf")
    plt.show()

In [ ]:
cluster_keys = ['leiden_4C_1', 'leiden_4C_2', 'leiden_4C_3', 'leiden_4C_4', 'leiden_4C_5']
scores = [silhouette_score(adata.obsm['X_pca'], adata.obs[key]) for key in cluster_keys]
n_clusters = [adata.obs[key].nunique() for key in cluster_keys]

fig, axes = plt.subplots(2, 1, figsize=(4, 7), sharex=True)

axes[0].plot(cluster_keys, scores, marker='o', color='#4066D9')
axes[0].set_ylabel('Silhouette Score', fontsize=14)
axes[0].tick_params(axis='both', labelsize=12)
axes[0].set_ylim(-0.25, 0)
axes[0].grid(False)
axes[0].grid(axis='y', zorder=0)

axes[1].bar(cluster_keys, n_clusters, color='#4066D9', zorder=3)
axes[1].set_ylabel('Number of Clusters', fontsize=14)
axes[1].set_xlabel('Resolution', fontsize=14)
axes[1].tick_params(axis='both', labelsize=12)
axes[1].grid(False)
axes[1].grid(axis='y', zorder=0)
axes[1].set_yticks([0, 25, 50, 75, 100, 125])
axes[1].set_ylim(0, 125)

plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig('./Figure_plots/'+ figure + '/2609/' + figure + name_of_subclustering + "_silhouette.pdf")
plt.show()

# 4CnCL (neoblast clusters) subclustering analysis

In [ ]:
name_of_subclustering = '_4CnCL_'
adata = sc.read_h5ad('./h5ad/analysis_250528_f/Smed_L47_Subclustering_4CnCL_Results.h5ad')

In [ ]:
sc.pl.umap(adata, color = 'annotated_names',  legend_loc = None, frameon = False, title = 'Subclustering' + name_of_subclustering, save = name_of_subclustering)

In [ ]:
results = pd.DataFrame(
    0,
    index = names,
    columns = resolutions,
    dtype = int
)

In [ ]:
for i in resolutions:
    for l in adata.obs['leiden'+ name_of_subclustering + str(i)].cat.categories:
        counts = adata.obs.loc[(adata.obs['leiden'+ name_of_subclustering + str(i)] == l), 'annotated_names'].value_counts().sort_values(ascending = False)
        if counts[0] > (counts[1:].sum() / 10):
            results.loc[counts.index[0], i] += 1

In [ ]:
cmap = LinearSegmentedColormap.from_list('custom', 
    [(0, (0,0,0)), (0.01, (0.4, 0.1, 0.5)), (0.15, (0.25, 0.4, 0.85)), (0.4, (0.45, 0.65, 0.95)), (1, (0.88, 0.91, 0.95))])

In [ ]:
fig, ax = plt.subplots(figsize=(13,30))
sns.heatmap(results.iloc[:-5, :],cmap = cmap, linewidths=1, linecolor='grey', ax=ax)
plt.grid(False)


for ticklabel, color in zip(ax.get_yticklabels(), colours_n):
    ticklabel.set_color(color)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=25)

plt.yticks(fontsize = 25)
plt.xticks(fontsize = 30)
    
plt.tight_layout()
plt.savefig('./Figure_plots/'+ figure + '/2609/' + figure + name_of_subclustering + "_heatmap.pdf")    
plt.show()

In [ ]:
with plt.rc_context({'figure.figsize': (2, 5)}):
    ax = sc.pl.violin(adata, keys='n_counts', size=0.5, jitter=0.4, show=False)
    ax.grid(zorder=0)
    ax.set_axisbelow(True)
    plt.axhline(y=adata.obs['n_counts'].mean(), color='r', linestyle='-')
    plt.savefig('./Figure_plots/'+ figure + '/2609/' + figure + name_of_subclustering + "_violin.pdf")
    plt.show()

In [ ]:
cluster_keys = ['leiden_4CnCL_1', 'leiden_4CnCL_2', 'leiden_4CnCL_3', 'leiden_4CnCL_4', 'leiden_4CnCL_5']

In [ ]:
scores = [silhouette_score(adata.obsm['X_pca'], adata.obs[key]) for key in cluster_keys]
n_clusters = [adata.obs[key].nunique() for key in cluster_keys]

fig, axes = plt.subplots(2, 1, figsize=(4, 7), sharex=True)

axes[0].plot(cluster_keys, scores, marker='o', color='#4066D9')
axes[0].set_ylabel('Silhouette Score', fontsize=14)
axes[0].tick_params(axis='both', labelsize=12)
axes[0].set_ylim(-0.25, 0)
axes[0].grid(False)
axes[0].grid(axis='y', zorder=0)

axes[1].bar(cluster_keys, n_clusters, color='#4066D9', zorder=3)
axes[1].set_ylabel('Number of Clusters', fontsize=14)
axes[1].set_xlabel('Resolution', fontsize=14)
axes[1].tick_params(axis='both', labelsize=12)
axes[1].grid(False)
axes[1].grid(axis='y', zorder=0)
axes[1].set_yticks([0, 25, 50, 75, 100, 125])
axes[1].set_ylim(0, 125)

plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig('./Figure_plots/'+ figure + '/2609/' + figure + name_of_subclustering + "_silhouette.pdf")
plt.show()

# 4CnSC (neoblasts score) subclustering analysis

In [ ]:
name_of_subclustering = '_4CnSC_'
adata = sc.read_h5ad('./h5ad/analysis_250528_f/test subclustering 2609/Smed_L47_Subclustering_4CnSC_Results_2608.h5ad')

In [ ]:
sc.pl.umap(adata, color = 'annotated_names',  legend_loc = None, frameon = False, title = 'Subclustering' + name_of_subclustering, save = name_of_subclustering)

In [ ]:
results = pd.DataFrame(
    0,
    index = names,
    columns = resolutions,
    dtype = int
)

In [ ]:
for i in resolutions:
    for l in adata.obs['leiden'+ name_of_subclustering + str(i)].cat.categories:
        counts = adata.obs.loc[(adata.obs['leiden'+ name_of_subclustering + str(i)] == l), 'annotated_names'].value_counts().sort_values(ascending = False)
        if counts[0] > (counts[1:].sum() / 10):
            results.loc[counts.index[0], i] += 1

In [ ]:
cmap = LinearSegmentedColormap.from_list('custom', 
    [(0, (0,0,0)), (0.01, (0.4, 0.1, 0.5)), (0.15, (0.25, 0.4, 0.85)), (0.4, (0.45, 0.65, 0.95)), (1, (0.88, 0.91, 0.95))])

In [ ]:
fig, ax = plt.subplots(figsize=(13,30))
sns.heatmap(results.iloc[:-5, :],cmap = cmap, linewidths=1, linecolor='grey', ax=ax)
plt.grid(False)


for ticklabel, color in zip(ax.get_yticklabels(), colours_n):
    ticklabel.set_color(color)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=25)

plt.yticks(fontsize = 25)
plt.xticks(fontsize = 30)
    
plt.tight_layout()
plt.savefig('./Figure_plots/'+ figure + '/2609/' + figure + name_of_subclustering + "_heatmap.pdf")    
plt.show()

In [ ]:
adata.obs['n_counts'].mean()

In [ ]:
with plt.rc_context({'figure.figsize': (2, 5)}):
    ax = sc.pl.violin(adata, keys='n_counts', size=0.5, jitter=0.4, show=False)
    ax.grid(zorder=0)
    ax.set_axisbelow(True)
    plt.axhline(y=adata.obs['n_counts'].mean(), color='r', linestyle='-')
    plt.savefig('./Figure_plots/'+ figure + '/2609/' + figure + name_of_subclustering + "_violin.pdf")
    plt.show()

In [ ]:
adata.obs.columns

In [ ]:
cluster_keys = ['leiden_4CnSC_1', 'leiden_4CnSC_2', 'leiden_4CnSC_3', 'leiden_4CnSC_4', 'leiden_4CnSC_5']
scores = [silhouette_score(adata.obsm['X_pca'], adata.obs[key]) for key in cluster_keys]
n_clusters = [adata.obs[key].nunique() for key in cluster_keys]

fig, axes = plt.subplots(2, 1, figsize=(4, 7), sharex=True)

axes[0].plot(cluster_keys, scores, marker='o', color='#4066D9')
axes[0].set_ylabel('Silhouette Score', fontsize=14)
axes[0].tick_params(axis='both', labelsize=12)
axes[0].set_ylim(-0.25, 0)
axes[0].grid(False)
axes[0].grid(axis='y', zorder=0)

axes[1].bar(cluster_keys, n_clusters, color='#4066D9', zorder=3)
axes[1].set_ylabel('Number of Clusters', fontsize=14)
axes[1].set_xlabel('Resolution', fontsize=14)
axes[1].tick_params(axis='both', labelsize=12)
axes[1].grid(False)
axes[1].grid(axis='y', zorder=0)
axes[1].set_yticks([0, 25, 50, 75, 100, 125])
axes[1].set_ylim(0, 125)

plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig('./Figure_plots/'+ figure + '/2609/' + figure + name_of_subclustering + "_silhouette.pdf")
plt.show()